## Problem 2-1

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# load the dataset
data = pd.read_csv('/Users/mengchuishuo/Desktop/5054/HW1/Life Expectancy Data.csv')

In [2]:
# Linear regression model : life expectancy ~ predicting variables except Country
data = data.dropna() #delete the NaN
X = data.drop(columns=["Life expectancy ", "Country"])
status_mapping = {"Developing": 1, "Developed": 2}
X["Status"] = X["Status"].map(status_mapping) #convert to numeric value
X = sm.add_constant(X) # Adds a constant term to the predictor
Y = data["Life expectancy "]


model = sm.OLS(Y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       Life expectancy    R-squared:                       0.839
Model:                            OLS   Adj. R-squared:                  0.837
Method:                 Least Squares   F-statistic:                     422.9
Date:                Tue, 23 Sep 2025   Prob (F-statistic):               0.00
Time:                        11:31:51   Log-Likelihood:                -4421.2
No. Observations:                1649   AIC:                             8884.
Df Residuals:                    1628   BIC:                             8998.
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

### According to the summary of model, observing the p-value. We can find Year, Status, Adult Mortality, infant deaths, Alcohol, BMI, under-five deaths, Total expenditure, Diphtheria, HIV/AIDS, Income composition of resources and Schooling are actually affecting the life expectancy.

## Problem 2-2

### Yes. According to the summary of model, observing the confidence interval. The coefficient of “Adult Mortality” and “HIV/AIDS” in 95% confidence interval are all negative. Adult Mortality is from -0.018 to -0.014. HIV/AIDS is from -0.483 to -0.413. Therefore, these predictors have negative impact on the life expectancy.

## Problem 2-3

In [3]:
ci = model.conf_int(alpha=0.03).loc[["Schooling", "Alcohol"]]

alpha = 0.03 # construct confidence interval is 97%
lower_quantile = alpha/2
upper_quantile = 1 - alpha/2
ci.columns = [f"{lower_quantile:.3f}", f"{upper_quantile:.3f}"]
ci

,0.015,0.985
Schooling,0.766518,1.023247
Alcohol,-0.204419,-0.058210


### According to the confidence interval. The coefficient of “Schooling” in 97% confidence interval is positive from 0.766518 to 1.023247. The coefficient of “Alcohol” in 97% confidence interval is negative from -0.204419 to -0.058210. Therefore, Schooling has positive impact on the life expectancy. And Alcohol has negative impact on the life expectancy.

## Problem 2-4

### When multiple variables have a p - value of 0.000, the t - value is used to distinguish them. The larger the absolute value of the t - value, the stronger the explanatory power of the variable for the dependent variable. Therefore, the top-seven most influential predictors are HIV/AIDS, Adult Mortality, Schooling, Income composition of resources, under-five deaths, infant deaths, Year.

In [4]:
X_smaller = data[[" HIV/AIDS", "Adult Mortality", "Schooling", 
          "Income composition of resources", 
          "under-five deaths ", "infant deaths", "Year"]] # choose the 7-top influential predictors

X_smaller = sm.add_constant(X_smaller)


Y_smaller = data["Life expectancy "]

model_smaller = sm.OLS(Y_smaller, X_smaller).fit()

print(model_smaller.summary())

                            OLS Regression Results                            
Dep. Variable:       Life expectancy    R-squared:                       0.824
Model:                            OLS   Adj. R-squared:                  0.823
Method:                 Least Squares   F-statistic:                     1096.
Date:                Tue, 23 Sep 2025   Prob (F-statistic):               0.00
Time:                        11:31:51   Log-Likelihood:                -4493.3
No. Observations:                1649   AIC:                             9003.
Df Residuals:                    1641   BIC:                             9046.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

## Problem 2-5

In [5]:
new_obs = pd.DataFrame({
    "const": [1],   
    "HIV/AIDS": [0.5],
    "Adult Mortality": [125],
    "Schooling": [18],
    "Income composition of resources": [0.9],
    "under-five deaths": [2],
    "infant deaths": [94],
    "Year": [2008]
})

pred = model_smaller.get_prediction(new_obs)


pred_summary = pred.summary_frame(alpha=0.01)  
print(pred_summary)

       mean   mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  \
0  88.38095  0.926114       85.99266       90.76924     78.544814   

   obs_ci_upper  
0     98.217086  


## Problem 2-6

### The AIC of full model is 8884, and the AIC of smaller model is 9003. The model with the smallest AIC is preferred. Therefore, th full mode is better to fit the data. However, the smaller model is also attractive because it achives the similar R^2 with few predictors, and smaller model is easy to interpret

## Problem 3

In [6]:
# load the dataset firstly
train = pd.read_csv("/Users/mengchuishuo/Desktop/5054/HW1/boston_housing_train.csv")
test = pd.read_csv("/Users/mengchuishuo/Desktop/5054/HW1/boston_housing_test.csv")

X_train = train.drop(columns=["medv"]).values
y_train = train["medv"].values

X_test = test.drop(columns=["medv"]).values
y_test = test["medv"].values

In [7]:
# construct KNN instead of directly using KNN in Python
class KNN:
    def knn_predict(self, X_train, y_train, X_test, K):
        y_pred = []
        for X in X_test:
            distances = np.sqrt(np.sum((X_train - X) ** 2, axis = 1))
            neighbors = np.argsort(distances)[:K]
            pred = np.mean(y_train[neighbors])
            y_pred.append(pred)
        return np.array(y_pred)
    
    def mse(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

## Problem 3-1

In [8]:
import time
import numpy as np
Ks = list(range(1,21))
knn = KNN()
results_no_std = []
for K in Ks:
    start = time.time()
    y_pred = knn.knn_predict(X_train, y_train, X_test, K)
    end = time.time()
    mse_val = knn.mse(y_test, y_pred)
    results_no_std.append((K, mse_val, end - start))

In [9]:
results_no_std

[(1, 44.51708661417323, 0.0061190128326416016),
 (2, 46.05590551181103, 0.004860877990722656),
 (3, 41.52323709536307, 0.00481724739074707),
 (4, 40.88791830708661, 0.004821062088012695),
 (5, 42.24112440944882, 0.004808902740478516),
 (6, 43.889407261592304, 0.004820108413696289),
 (7, 43.985068295034544, 0.0048520565032958984),
 (8, 42.83032357283464, 0.00480198860168457),
 (9, 44.04344026441139, 0.004803180694580078),
 (10, 45.6143283464567, 0.00481104850769043),
 (11, 45.783520531007994, 0.0048220157623291016),
 (12, 45.87835465879264, 0.004805803298950195),
 (13, 45.76500628989423, 0.004812955856323242),
 (14, 46.51385987465852, 0.004807949066162109),
 (15, 46.5348738407699, 0.004806041717529297),
 (16, 48.18960999015748, 0.004801034927368164),
 (17, 49.132718851320035, 0.009046077728271484),
 (18, 48.92646058131623, 0.0056498050689697266),
 (19, 50.128822823739824, 0.005947113037109375),
 (20, 51.09004133858268, 0.005736827850341797)]

### According to mse and time, we can find the best K value is 4. Whe K = 4, model has least mse and the short time.

## Problem 3-2

In [10]:
# standardize the predictor
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_std = (X_train - mean) / std
X_test_std = (X_test - mean) / std

results_std = []
for K in Ks:
    start = time.time()
    y_pred = knn.knn_predict(X_train_std, y_train, X_test_std, K)
    end = time.time()
    mse_val = knn.mse(y_test, y_pred)
    results_std.append((K, mse_val, end - start))

In [11]:
results_std

[(1, 25.46929133858267, 0.008126974105834961),
 (2, 16.777578740157477, 0.006354808807373047),
 (3, 19.73487314085739, 0.005498170852661133),
 (4, 20.019940944881885, 0.005663156509399414),
 (5, 21.226831496062992, 0.005071163177490234),
 (6, 22.255448381452318, 0.005997896194458008),
 (7, 21.654661738711233, 0.005419731140136719),
 (8, 20.868348917322834, 0.005616188049316406),
 (9, 21.036780402449686, 0.0055408477783203125),
 (10, 20.50813543307087, 0.005807161331176758),
 (11, 20.584192750699557, 0.005114078521728516),
 (12, 20.443277559055122, 0.005421161651611328),
 (13, 20.855489912873313, 0.005808830261230469),
 (14, 21.566415715892653, 0.005814075469970703),
 (15, 21.72329658792652, 0.005722999572753906),
 (16, 22.093315083661416, 0.005117893218994141),
 (17, 22.66307250088548, 0.004971742630004883),
 (18, 23.372526489744338, 0.00496220588684082),
 (19, 24.198553449516872, 0.004970073699951172),
 (20, 24.45389448818897, 0.0049610137939453125)]

### According to mse and time, after standardize the predictor variables, we can find the best K value is 2. Whe K = 2, model has least mse and the short time.

## Problem 3-3

### Yes, standardization improve the performance. After standardization, the test mse descends from 40.88791830708661 to 16.777578740157477, we can see it drops significantly. While the running time remains nearly same. Therefore, i think standardization improve the performance.